# Day 078 — Solution: Multimodal Media Studio

In [ ]:
_SRC = '"""media_studio.py — Day 078: Capstone — Multimodal Media Studio.\n\nCombines Section 5 skills into a unified media processing interface:\n  detect_media_type  — identify file type from extension\n  describe_media     — image -> text description (llava via Ollama)\n  transcribe_media   — audio -> text transcript (Whisper)\n  synthesize_speech  — text -> audio MP3 bytes (edge-tts)\n  narrate_image      — image -> description -> audio\n  process_media      — auto-detect type and dispatch\n  MediaStudio        — stateful class with all capabilities + batch\n\nSetup:\n    pip install pillow ollama openai-whisper edge-tts\n    ollama pull llava\n"""\nimport io\nimport base64\nimport asyncio\nfrom pathlib import Path\n\nMEDIA_EXTENSIONS = {\n    \'image\': {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.gif\', \'.webp\', \'.tiff\'},\n    \'audio\': {\'.mp3\', \'.wav\', \'.ogg\', \'.flac\', \'.m4a\', \'.aac\'},\n    \'video\': {\'.mp4\', \'.avi\', \'.mov\', \'.mkv\', \'.webm\', \'.flv\'},\n}\n\n\ndef detect_media_type(path):\n    """Return media type string from file extension.\n\n    Args:\n        path: file path (str or Path)\n    Returns:\n        \'image\', \'audio\', \'video\', or \'unknown\'\n    """\n    ext = Path(path).suffix.lower()\n    for media_type, extensions in MEDIA_EXTENSIONS.items():\n        if ext in extensions:\n            return media_type\n    return \'unknown\'\n\n\ndef describe_media(source, describe_fn=None):\n    """Describe an image using a vision LLM.\n\n    Args:\n        source:     PIL Image or file path (str/Path)\n        describe_fn: callable(pil_image, prompt) -> str for testing\n    Returns:\n        description string\n    """\n    from PIL import Image\n    if isinstance(source, (str, Path)):\n        image = Image.open(source)\n    else:\n        image = source\n    prompt = \'Describe this image in detail, including all visible content.\'\n    if describe_fn is not None:\n        return describe_fn(image, prompt)\n    import ollama\n    buf = io.BytesIO()\n    image.save(buf, format=\'PNG\')\n    img_b64 = base64.b64encode(buf.getvalue()).decode()\n    resp = ollama.chat(\n        model=\'llava\',\n        messages=[{\'role\': \'user\', \'content\': prompt, \'images\': [img_b64]}],\n    )\n    return resp[\'message\'][\'content\']\n\n\ndef transcribe_media(source, transcribe_fn=None):\n    """Transcribe audio to text using Whisper.\n\n    Args:\n        source:        audio bytes or file path (str/Path)\n        transcribe_fn: callable(source) -> dict for testing\n    Returns:\n        dict with keys: text (str), segments (list of {start, end, text})\n    """\n    if transcribe_fn is not None:\n        return transcribe_fn(source)\n    import whisper, os, tempfile\n    model = whisper.load_model(\'base\')\n    if isinstance(source, bytes):\n        with tempfile.NamedTemporaryFile(suffix=\'.wav\', delete=False) as f:\n            f.write(source)\n            tmp = f.name\n        try:\n            raw = model.transcribe(tmp)\n        finally:\n            os.unlink(tmp)\n    else:\n        raw = model.transcribe(str(source))\n    segments = [\n        {\'start\': s[\'start\'], \'end\': s[\'end\'], \'text\': s[\'text\'].strip()}\n        for s in raw.get(\'segments\', [])\n    ]\n    return {\'text\': raw.get(\'text\', \'\').strip(), \'segments\': segments}\n\n\ndef synthesize_speech(text, voice=\'en-US-AriaNeural\', rate=\'+0%\', pitch=\'+0Hz\',\n                      tts_fn=None):\n    """Convert text to speech and return MP3 bytes.\n\n    Args:\n        text:   text to synthesize\n        voice:  edge-tts voice ShortName (e.g. \'en-US-AriaNeural\')\n        rate:   speed adjustment (\'+0%\', \'+20%\', \'-20%\')\n        pitch:  pitch adjustment (\'+0Hz\', \'+5Hz\', \'-5Hz\')\n        tts_fn: callable(text, voice, rate, pitch) -> bytes for testing\n    Returns:\n        MP3 bytes\n    """\n    if tts_fn is not None:\n        return tts_fn(text, voice, rate, pitch)\n    import edge_tts\n    async def _run():\n        chunks = []\n        async for chunk in edge_tts.Communicate(\n                text, voice, rate=rate, pitch=pitch).stream():\n            if chunk[\'type\'] == \'audio\':\n                chunks.append(chunk[\'data\'])\n        return b\'\'.join(chunks)\n    return asyncio.run(_run())\n\n\ndef narrate_image(source, voice=\'en-US-AriaNeural\', describe_fn=None, tts_fn=None):\n    """Describe an image then convert the description to speech.\n\n    Args:\n        source:     PIL Image or file path\n        voice:      edge-tts voice ShortName\n        describe_fn: callable(pil_image, prompt) -> str for testing\n        tts_fn:      callable(text, voice, rate, pitch) -> bytes for testing\n    Returns:\n        dict with keys: description (str), audio (bytes)\n    """\n    description = describe_media(source, describe_fn=describe_fn)\n    audio = synthesize_speech(description, voice=voice, tts_fn=tts_fn)\n    return {\'description\': description, \'audio\': audio}\n\n\ndef process_media(source, describe_fn=None, transcribe_fn=None, tts_fn=None):\n    """Auto-detect media type and process the file.\n\n    - image  -> describe_media -> str description\n    - audio  -> transcribe_media -> {text, segments}\n    - other  -> {note: \'...\'}\n\n    Returns:\n        dict with keys: type (str), source (str), result\n    """\n    media_type = detect_media_type(source)\n    if media_type == \'image\':\n        result = describe_media(source, describe_fn=describe_fn)\n    elif media_type == \'audio\':\n        result = transcribe_media(source, transcribe_fn=transcribe_fn)\n    else:\n        result = {\'note\': f\'Media type {media_type!r} detected but not processed\'}\n    return {\'type\': media_type, \'source\': str(source), \'result\': result}\n\n\nclass MediaStudio:\n    """Multimodal media processing studio.\n\n    Combines image description, audio transcription, and text-to-speech\n    synthesis into a single injectable interface.\n\n    Example::\n\n        studio = MediaStudio(\n            describe_fn=lambda img, q: \'Mock description\',\n            transcribe_fn=lambda src: {\'text\': \'Mock transcript\', \'segments\': []},\n            tts_fn=lambda text, v, r, p: b\'AUDIO\',\n        )\n        desc    = studio.describe(image)\n        text    = studio.transcribe(audio_bytes)[\'text\']\n        audio   = studio.speak(\'Hello world\')\n        result  = studio.narrate(image)\n        results = studio.batch([\'image.png\', \'audio.wav\'])\n    """\n\n    def __init__(self, describe_fn=None, transcribe_fn=None, tts_fn=None):\n        self._describe_fn = describe_fn\n        self._transcribe_fn = transcribe_fn\n        self._tts_fn = tts_fn\n\n    def describe(self, source):\n        """Describe an image (PIL Image or path)."""\n        return describe_media(source, describe_fn=self._describe_fn)\n\n    def transcribe(self, source):\n        """Transcribe audio (bytes or path). Returns {text, segments}."""\n        return transcribe_media(source, transcribe_fn=self._transcribe_fn)\n\n    def speak(self, text, voice=\'en-US-AriaNeural\', rate=\'+0%\', pitch=\'+0Hz\'):\n        """Convert text to speech. Returns MP3 bytes."""\n        return synthesize_speech(text, voice=voice, rate=rate, pitch=pitch,\n                                 tts_fn=self._tts_fn)\n\n    def narrate(self, source, voice=\'en-US-AriaNeural\'):\n        """Describe an image and speak the description."""\n        return narrate_image(source, voice=voice,\n                             describe_fn=self._describe_fn, tts_fn=self._tts_fn)\n\n    def process(self, source):\n        """Auto-detect media type and process the file."""\n        return process_media(source, describe_fn=self._describe_fn,\n                             transcribe_fn=self._transcribe_fn, tts_fn=self._tts_fn)\n\n    def batch(self, sources):\n        """Process a list of media sources. Returns list of result dicts."""\n        return [self.process(s) for s in sources]\n'
from pathlib import Path
Path('media_studio.py').write_text(_SRC, encoding='utf-8')
print('media_studio.py written.')

In [ ]:

from pathlib import Path
from PIL import Image as PILImage
from media_studio import (
    MEDIA_EXTENSIONS, detect_media_type, describe_media, transcribe_media,
    synthesize_speech, narrate_image, process_media, MediaStudio,
)
import tempfile, os

_mock_describe_fn   = lambda img, q: 'A solid color test image.'
_mock_transcribe_fn = lambda src: {'text': 'Hello world.', 'segments': []}
_mock_tts_fn        = lambda t, v, r, p: b'AUDIO:' + t[:8].encode()

# 1. detect_media_type
assert detect_media_type('photo.png') == 'image'
assert detect_media_type('audio.mp3') == 'audio'
assert detect_media_type('clip.mp4')  == 'video'
assert detect_media_type('data.txt')  == 'unknown'
print("✅ detect_media_type")

# 2. describe_media
img = PILImage.new('RGB', (50, 50), color=(100,100,100))
d = describe_media(img, describe_fn=_mock_describe_fn)
assert isinstance(d, str) and len(d) > 0
print("✅ describe_media (PIL Image)")

with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    img.save(f, format='PNG'); tmp_img = f.name
try:
    d2 = describe_media(tmp_img, describe_fn=_mock_describe_fn)
    assert isinstance(d2, str)
    print("✅ describe_media (file path)")
finally:
    os.unlink(tmp_img)

# 3. transcribe_media
t = transcribe_media(b'AUDIO', transcribe_fn=_mock_transcribe_fn)
assert isinstance(t, dict) and 'text' in t and 'segments' in t
print("✅ transcribe_media")

# 4. synthesize_speech
audio = synthesize_speech('Hello', tts_fn=_mock_tts_fn)
assert isinstance(audio, bytes) and len(audio) > 0
print("✅ synthesize_speech")

# 5. narrate_image
n = narrate_image(img, describe_fn=_mock_describe_fn, tts_fn=_mock_tts_fn)
assert 'description' in n and 'audio' in n
assert isinstance(n['description'], str) and isinstance(n['audio'], bytes)
print("✅ narrate_image")

# 6. process_media
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    img.save(f, format='PNG'); tmp2 = f.name
try:
    r = process_media(tmp2, describe_fn=_mock_describe_fn)
    assert r['type'] == 'image' and isinstance(r['result'], str)
    print("✅ process_media (image)")
finally:
    os.unlink(tmp2)

r2 = process_media('test.mp3', transcribe_fn=_mock_transcribe_fn)
assert r2['type'] == 'audio' and 'text' in r2['result']
print("✅ process_media (audio)")

r3 = process_media('data.txt')
assert r3['type'] == 'unknown' and 'note' in r3['result']
print("✅ process_media (unknown)")

# 7. MediaStudio
studio = MediaStudio(describe_fn=_mock_describe_fn,
                     transcribe_fn=_mock_transcribe_fn,
                     tts_fn=_mock_tts_fn)
assert isinstance(studio.describe(img), str)
assert isinstance(studio.transcribe(b'A')['text'], str)
assert isinstance(studio.speak('Hello'), bytes)
narr = studio.narrate(img)
assert 'description' in narr and 'audio' in narr
results = studio.batch(['x.png', 'y.mp3', 'z.txt'])
assert len(results) == 3
print("✅ MediaStudio (describe/transcribe/speak/narrate/batch)")

print("\nMedia Studio complete! Section 5 done.")
